In [2]:
pip install pandas requests beautifulsoup4 lxml

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
import requests
from bs4 import BeautifulSoup

base_url = "https://en.wikipedia.org"
url = "https://en.wikipedia.org/wiki/List_of_Premier_League_seasons"

headers = {
    "User-Agent": "Mozilla/5.0"
}

res = requests.get(url, headers=headers)

print(res.status_code)   # should be 200
print(len(res.text))     # should NOT be small

soup = BeautifulSoup(res.text, "html.parser")

links = []

for link in soup.find_all("a"):
    href = link.get("href")
    
    if href and "Premier_League" in href:
        full_link = base_url + href
        
        if full_link not in links:
            links.append(full_link)

print(f"Total links found: {len(links)}")
print(links[:10])

200
273786
Total links found: 125
['https://en.wikipedia.orghttps://et.wikipedia.org/wiki/Premier_League%27i_hooaegade_loend', 'https://en.wikipedia.orghttps://sk.wikipedia.org/wiki/Zoznam_sez%C3%B3n_Premier_League', 'https://en.wikipedia.org/wiki/List_of_Premier_League_seasons', 'https://en.wikipedia.org/wiki/Talk:List_of_Premier_League_seasons', 'https://en.wikipedia.org/w/index.php?title=List_of_Premier_League_seasons&action=edit', 'https://en.wikipedia.org/w/index.php?title=List_of_Premier_League_seasons&action=history', 'https://en.wikipedia.org/wiki/Special:WhatLinksHere/List_of_Premier_League_seasons', 'https://en.wikipedia.org/wiki/Special:RecentChangesLinked/List_of_Premier_League_seasons', 'https://en.wikipedia.org/w/index.php?title=List_of_Premier_League_seasons&oldid=1358946199', 'https://en.wikipedia.org/w/index.php?title=List_of_Premier_League_seasons&action=info']


In [3]:
season_links = [
    link for link in links
    if (
        "Premier_League" in link
        and "%E2%80%93" in link   # ensures it's a season format (1992–93)
    )
]

print(f"Filtered season links: {len(season_links)}")
print(season_links[:10])

Filtered season links: 38
['https://en.wikipedia.org/wiki/2025%E2%80%9326_Premier_League', 'https://en.wikipedia.org/wiki/1992%E2%80%9393_FA_Premier_League', 'https://en.wikipedia.org/wiki/1993%E2%80%9394_FA_Premier_League', 'https://en.wikipedia.org/wiki/1994%E2%80%9395_FA_Premier_League', 'https://en.wikipedia.org/wiki/1995%E2%80%9396_FA_Premier_League', 'https://en.wikipedia.org/wiki/1996%E2%80%9397_FA_Premier_League', 'https://en.wikipedia.org/wiki/1997%E2%80%9398_FA_Premier_League', 'https://en.wikipedia.org/wiki/1999%E2%80%932000_FA_Premier_League', 'https://en.wikipedia.org/wiki/2000%E2%80%9301_FA_Premier_League', 'https://en.wikipedia.org/wiki/2001%E2%80%9302_FA_Premier_League']


In [4]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

test_url = season_links[0]

res = requests.get(test_url, headers=headers)

print(res.status_code)  # should be 200

tables = pd.read_html(res.text)

for i, table in enumerate(tables):
    print(f"\nTable {i}")
    print(table.head())

200

Table 0
                                                   0  \
0  Arsenal being crowned champions on 24 May 2026...   
1                                             Season   
2                                              Dates   
3                                          Champions   
4                                          Relegated   

                                                   1  
0  Arsenal being crowned champions on 24 May 2026...  
1                                            2025–26  
2                       15 August 2025 – 24 May 2026  
3  Arsenal 4th Premier League title 14th English ...  
4    West Ham United Burnley Wolverhampton Wanderers  

Table 1
                     Team            Location                      Stadium  \
0                 Arsenal   London (Holloway)             Emirates Stadium   
1             Aston Villa          Birmingham                   Villa Park   
2             Bournemouth         Bournemouth                   Dean Court   

C:\Users\USER\AppData\Local\Temp\ipykernel_19344\1474135268.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(res.text)


In [5]:
for table in tables:
    cols = [str(col).lower() for col in table.columns]
    
    if (
        'pos' in cols
        and ('team' in cols or 'club' in cols)
        and ('pts' in cols or 'points' in cols)
    ):
        df = table.copy()
        break

In [6]:
df.columns = [str(col).lower() for col in df.columns]

team_col = 'team' if 'team' in df.columns else 'club'

# keep important columns
keep_cols = ['pos', team_col, 'pld', 'w', 'd', 'l', 'gf', 'ga', 'gd', 'pts']

# some seasons may not have exact same names → filter safely
keep_cols = [col for col in keep_cols if col in df.columns]

df = df[keep_cols]

# rename columns
df.rename(columns={team_col: 'team'}, inplace=True)

# clean position
df['pos'] = df['pos'].astype(str).str.extract('(\d+)').astype(int)

# clean team names
df['team'] = df['team'].str.replace(r"\s*\(.*\)", "", regex=True)

In [7]:
import pandas as pd
import requests

headers = {"User-Agent": "Mozilla/5.0"}

all_data = []

for link in season_links:
    try:
        res = requests.get(link, headers=headers)
        tables = pd.read_html(res.text)
        
        for table in tables:
            cols = [str(col).lower() for col in table.columns]
            
            # detect league table
            if (
                'pos' in cols
                and ('team' in cols or 'club' in cols)
                and ('pts' in cols or 'points' in cols)
            ):
                df = table.copy()
                df.columns = cols
                
                team_col = 'team' if 'team' in cols else 'club'
                
                # keep important columns
                keep_cols = ['pos', team_col, 'pld', 'w', 'd', 'l', 'gf', 'ga', 'gd', 'pts']
                keep_cols = [col for col in keep_cols if col in cols]
                
                df = df[keep_cols]
                df.rename(columns={team_col: 'team'}, inplace=True)
                
                # clean position
                df['pos'] = df['pos'].astype(str).str.extract('(\d+)').astype(int)
                
                # clean team names
                df['team'] = df['team'].str.replace(r"\s*\(.*\)", "", regex=True)
                
                # extract season
                season = link.split("/")[-1].split("_")[0]
                df['season'] = season
                
                all_data.append(df)
                print(f"Done: {season}")
                break
                
    except Exception as e:
        print(f"Failed: {link}")

C:\Users\USER\AppData\Local\Temp\ipykernel_19344\2242146431.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(res.text)


Done: 2025%E2%80%9326


C:\Users\USER\AppData\Local\Temp\ipykernel_19344\2242146431.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(res.text)


Done: 1992%E2%80%9393


C:\Users\USER\AppData\Local\Temp\ipykernel_19344\2242146431.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(res.text)


...

C:\Users\USER\AppData\Local\Temp\ipykernel_19344\2242146431.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(res.text)


Done: 2024%E2%80%9325


C:\Users\USER\AppData\Local\Temp\ipykernel_19344\2242146431.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(res.text)


Done: 2026%E2%80%9327


C:\Users\USER\AppData\Local\Temp\ipykernel_19344\2242146431.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(res.text)


In [8]:
final_df = pd.concat(all_data, ignore_index=True)

print(final_df.head())
print(final_df.shape)

   pos               team  pld   w   d   l  gf  ga   gd pts           season
0    1            Arsenal   38  26   7   5  71  27  +44  85  2025%E2%80%9326
1    2    Manchester City   38  23   9   6  77  35  +42  78  2025%E2%80%9326
2    3  Manchester United   38  20  11   7  69  50  +19  71  2025%E2%80%9326
3    4        Aston Villa   38  19   8  11  56  49   +7  65  2025%E2%80%9326
4    5          Liverpool   38  17   9  12  63  53  +10  60  2025%E2%80%9326
(746, 11)


In [9]:
final_df['season'] = final_df['season'].str.replace('%E2%80%93', '-', regex=False)

In [10]:
final_df['gd'] = (
    final_df['gd']
    .astype(str)
    .str.replace(r"[^\d\-]", "", regex=True)  # keep only numbers and minus
    .astype(int)
)

In [11]:
print(final_df['gd'].head())
print(final_df['gd'].dtype)

0    44
1    42
2    19
3     7
4    10
Name: gd, dtype: int64
int64


In [12]:
final_df['pts'] = final_df['pts'].astype(str).str.replace(r'[^\d.]', '', regex=True)

# Now convert to int (or float first if there are decimals)
final_df['pts'] = final_df['pts'].astype(float).astype(int)

In [13]:
print(final_df.dtypes)

pos        int64
team      object
pld        int64
w          int64
d          int64
l          int64
gf         int64
ga         int64
gd         int64
pts        int64
season    object
dtype: object


In [14]:
print(final_df.head(50))
print(final_df.describe())

    pos                     team  pld   w   d   l  gf  ga  gd  pts   season
0     1                  Arsenal   38  26   7   5  71  27  44   85  2025-26
1     2          Manchester City   38  23   9   6  77  35  42   78  2025-26
2     3        Manchester United   38  20  11   7  69  50  19   71  2025-26
3     4              Aston Villa   38  19   8  11  56  49   7   65  2025-26
4     5                Liverpool   38  17   9  12  63  53  10   60  2025-26
5     6              Bournemouth   38  13  18   7  58  54   4   57  2025-26
6     7               Sunderland   38  14  12  12  42  48   6   54  2025-26
7     8   Brighton & Hove Albion   38  14  11  13  52  46   6   53  2025-26
8     9                Brentford   38  14  11  13  55  52   3   53  2025-26
9    10                  Chelsea   38  14  10  14  58  52   6   52  2025-26
10   11                   Fulham   38  15   7  16  47  51   4   52  2025-26
11   12         Newcastle United   38  14   7  17  53  55   2   49  2025-26
12   13     

In [15]:
print(final_df['season'].nunique())   # should be ~30+
print(final_df['team'].nunique())     # many teams
print(final_df.isnull().sum())        # check missing values

35
52
pos       0
team      0
pld       0
w         0
d         0
l         0
gf        0
ga        0
gd        0
pts       0
season    0
dtype: int64


In [32]:
# Define the file name and path
file_path = r'C:\Users\USER\OneDrive\Desktop\premier_league_table.csv'

# Save the dataframe
final_df.to_csv(file_path, index=False)

print(f"File saved successfully to {file_path}")

File saved successfully to C:\Users\USER\OneDrive\Desktop\premier_league_table.csv


In [16]:
final_df = final_df.sort_values(by=['season', 'pos'], ascending=[True, True])

# If you want the most recent seasons at the top, change season to False
# final_df = final_df.sort_values(by=['season', 'pos'], ascending=[False, True])

print(final_df.head(20))

    pos                 team  pld   w   d   l  gf  ga  gd  pts   season
20    1    Manchester United   42  24  12   6  67  31  36   84  1992-93
21    2          Aston Villa   42  21  11  10  57  40  17   74  1992-93
22    3         Norwich City   42  21   9  12  61  65   4   72  1992-93
23    4     Blackburn Rovers   42  20  11  11  68  46  22   71  1992-93
24    5  Queens Park Rangers   42  17  12  13  63  55   8   63  1992-93
25    6            Liverpool   42  16  11  15  62  55   7   59  1992-93
26    7  Sheffield Wednesday   42  15  14  13  55  51   4   59  1992-93
27    8    Tottenham Hotspur   42  16  11  15  60  66   6   59  1992-93
28    9      Manchester City   42  15  12  15  56  51   5   57  1992-93
29   10              Arsenal   42  15  11  16  40  38   2   56  1992-93
30   11              Chelsea   42  14  14  14  51  54   3   56  1992-93
31   12            Wimbledon   42  14  12  16  56  55   1   54  1992-93
32   13              Everton   42  15   8  19  53  55   2   53  

In [17]:
unique_seasons = sorted(final_df['season'].unique())

print(f"Your data covers {len(unique_seasons)} seasons:")
print(unique_seasons)

Your data covers 35 seasons:
['1992-93', '1993-94', '1994-95', '1995-96', '1996-97', '1997-98', '1998-99', '1999-2000', '2000-01', '2001-02', '2002-03', '2003-04', '2004-05', '2005-06', '2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26', '2026-27']


In [18]:
# Shows how many rows exist for each season
print(final_df['season'].value_counts().sort_index())

season
1992-93      22
1993-94      22
1994-95      22
1995-96      20
1996-97      20
1997-98      20
1998-99      40
1999-2000    20
2000-01      20
2001-02      40
2002-03      20
2003-04      20
2004-05      20
2005-06      20
2006-07      20
2007-08      20
2008-09      20
2009-10      20
2010-11      20
2011-12      20
2012-13      20
2013-14      20
2014-15      20
2015-16      20
2016-17      20
2017-18      20
2018-19      20
2019-20      20
2020-21      20
2021-22      20
2022-23      20
2023-24      20
2024-25      20
2025-26      20
2026-27      20
Name: count, dtype: int64


In [19]:
# This keeps only one entry if 'team' and 'season' are identical
final_df = final_df.drop_duplicates(subset=['team', 'season'])

# Now check the counts again to confirm
print(final_df['season'].value_counts().sort_index())

season
1992-93      22
1993-94      22
1994-95      22
1995-96      20
1996-97      20
1997-98      20
1998-99      20
1999-2000    20
2000-01      20
2001-02      20
2002-03      20
2003-04      20
2004-05      20
2005-06      20
2006-07      20
2007-08      20
2008-09      20
2009-10      20
2010-11      20
2011-12      20
2012-13      20
2013-14      20
2014-15      20
2015-16      20
2016-17      20
2017-18      20
2018-19      20
2019-20      20
2020-21      20
2021-22      20
2022-23      20
2023-24      20
2024-25      20
2025-26      20
2026-27      20
Name: count, dtype: int64


In [20]:
# 1. Definitely drop the future season
final_df = final_df[final_df['season'] != '2026-27']

# 2. Option: Drop the ongoing season if you only want finished years
# final_df = final_df[final_df['season'] != '2025-26']

# 3. Double-check your counts one last time
print(final_df['season'].value_counts().sort_index())

season
1992-93      22
1993-94      22
1994-95      22
1995-96      20
1996-97      20
1997-98      20
1998-99      20
1999-2000    20
2000-01      20
2001-02      20
2002-03      20
2003-04      20
2004-05      20
2005-06      20
2006-07      20
2007-08      20
2008-09      20
2009-10      20
2010-11      20
2011-12      20
2012-13      20
2013-14      20
2014-15      20
2015-16      20
2016-17      20
2017-18      20
2018-19      20
2019-20      20
2020-21      20
2021-22      20
2022-23      20
2023-24      20
2024-25      20
2025-26      20
Name: count, dtype: int64


In [21]:
file_path = r'C:\Users\USER\OneDrive\Desktop\complete_premier_league_table_1993_to_2026.csv'

# Save the dataframe
final_df.to_csv(file_path, index=False)

print(f"File saved successfully to {file_path}")

File saved successfully to C:\Users\USER\OneDrive\Desktop\complete_premier_league_table_1993_to_2026.csv
